# Seminar 09: Normalizing Flows

In this notebook, we explore **normalizing flows**—a powerful class of generative models that transform a simple base distribution into a complex target distribution using a sequence of invertible transformations.

The code is based on https://github.com/karpathy/pytorch-normalizing-flows.

Date: 2025-03-11

## Notebook Structure
1. **Imports and Setup:** Load libraries and set up device management.
2. **Dataset Definitions:** Define toy datasets for visualization (e.g., half-moons, mixtures).
3. **Model Construction:** Build the normalizing flow model with a chosen sequence of flow layers.
4. **Training Loop:** Train the model by maximizing the log-likelihood.
5. **Visualization:** Visualize both the forward (data $\to$ latent) and inverse (latent $\to$ data) mappings.
6. **Detailed Grid Warp Visualization:** Inspect intermediate flow transformations.

## Mathematical Background

Let $z_0$ be a random variable with a known, simple probability density $p(z_0)$ (e.g., Gaussian or Logistic). We define a sequence of invertible transformations:

$$
z_k = f_k \circ f_{k-1} \circ \cdots \circ f_1(z_0)
$$

By the change-of-variables formula, the density of $z_k$ can be expressed as:

$$
p(z_k) = p(z_0) \prod_{i=1}^k \left| \det \left( \frac{\partial f_i}{\partial z_{i-1}} \right) \right|^{-1},
$$

or in log-space:

$$
\log p(z_k) = \log p(z_0) - \sum_{i=1}^k \log \left| \det \left( \frac{\partial f_i}{\partial z_{i-1}} \right) \right|.
$$

This framework allows us to design flexible models where the Jacobian determinants $J_i = \left| \det \left( \frac{\partial f_i}{\partial z_{i-1}} \right) \right|$ account for the local volume changes induced by the transformations.

## Types of Flow Transformations

Some popular flow transformations include:

- **Planar Flows:** $f(z) = z + u \, h(w^\top z + b)$
- **Radial Flows:** $f(z) = z + \frac{\beta}{\alpha + \lVert z - z_0 \rVert}(z - z_0)$
- **Real NVP:** Uses affine coupling layers: $f(x^{(2)}) = t(x^{(1)}) + x^{(2)} \odot \exp(s(x^{(1)}))$
- **Masked Autoregressive Flow (MAF):** $f(x_i) = \frac{x_i - \mu(x_{<i})}{\exp(\alpha(x_{<i}))}$
- **Invertible 1x1 Convolution (Glow)**
- **ActNorm:** $f(x) = Wx + b$ (with a diagonal $W$ and constant $b$)
- **Neural Spline Flows (NSF):** Use rational quadratic splines to model complex nonlinearities.

In [ ]:
import itertools
import pickle
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
from torch.distributions import MultivariateNormal, Uniform, TransformedDistribution, SigmoidTransform
from tqdm.auto import tqdm

# Import flow modules from nflib
from nflib.flows import (
    AffineConstantFlow, ActNorm, AffineHalfFlow, 
    SlowMAF, MAF, IAF, Invertible1x1Conv,
    NormalizingFlow, NormalizingFlowModel,
)
from nflib.spline_flows import NSF_AR, NSF_CL

# Set device
cuda = torch.cuda.is_available()
# device = torch.device("cuda" if cuda else ("mps" if torch.backends.mps.is_available() else "cpu"))
device = torch.device("cpu")
torch.backends.cudnn.benchmark = cuda

print(f"Using device: {device}")

## Dataset Definitions

We define several toy datasets to experiment with normalizing flows:

- **DatasetMoons:** Two interleaving half-moons.
- **DatasetMixture:** A mixture of four 2D Gaussians.
- **DatasetSIGGRAPH:** A dataset from SIGGRAPH (requires a `siggraph.pkl` file).

Each dataset class has a `sample(n)` method which returns `n` samples as a torch tensor.

In [ ]:
from sklearn import datasets

class DatasetSIGGRAPH:
    """
    SIGGRAPH dataset.
    
    Loads and centers the dataset from a 'siggraph.pkl' file.
    
    Raises:
        FileNotFoundError: If the 'siggraph.pkl' file is not found.
    """
    def __init__(self):
        try:
            with open('siggraph.pkl', 'rb') as f:
                # Load data and convert to numpy array of type float32
                XY = np.array(pickle.load(f), dtype=np.float32)
                XY -= np.mean(XY, axis=0)  # center the data
            self.XY = torch.from_numpy(XY)
        except FileNotFoundError:
            raise FileNotFoundError("The file 'siggraph.pkl' was not found. Please ensure it is in the working directory.")
    
    def sample(self, n):
        """
        Randomly sample n data points from the dataset.
        
        Args:
            n (int): Number of samples to return.
            
        Returns:
            torch.Tensor: Sampled data points.
        """
        indices = np.random.randint(self.XY.shape[0], size=n)
        return self.XY[indices]

class DatasetMoons:
    """
    Generates two interleaving half-moons with noise.
    
    Uses sklearn.datasets.make_moons to generate the data.
    """
    def sample(self, n):
        # Generate moons with added noise and convert to torch tensor
        moons = datasets.make_moons(n_samples=n, noise=0.05)[0].astype(np.float32)
        return torch.from_numpy(moons)

class DatasetMixture:
    """
    Generates a mixture of 4 Gaussians in 2D.
    
    Ensures the number of samples (n) is divisible by 4.
    """
    def sample(self, n):
        assert n % 4 == 0, "n must be divisible by 4"
        r = np.r_[np.random.randn(n // 4, 2) * 0.5 + np.array([0, -2]),
                  np.random.randn(n // 4, 2) * 0.5 + np.array([0, 0]),
                  np.random.randn(n // 4, 2) * 0.5 + np.array([2, 2]),
                  np.random.randn(n // 4, 2) * 0.5 + np.array([-2, 2])]
        return torch.from_numpy(r.astype(np.float32))

# Choose dataset (uncomment one as needed)
d = DatasetMoons()
# d = DatasetMixture()
# d = DatasetSIGGRAPH()

# Visualize the chosen dataset with enhanced plot labels and grid
x = d.sample(128)
plt.figure(figsize=(4, 4))
plt.scatter(x[:, 0], x[:, 1], s=5, alpha=0.5)
plt.title("Toy Dataset Samples", fontsize=12)
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.grid(True)
plt.axis('equal')
plt.show()

## Model Construction

Here, we construct the normalizing flow model:
- **Prior:** A logistic distribution obtained via an inverse Sigmoid transform on a Uniform distribution.
- **Flows:** A sequence of flow layers (here using Neural Spline Flow Coupling layers in coupling mode (NSF_CL)) interleaved with invertible 1x1 convolutions and ActNorm layers.

The final normalizing flow model is constructed as:

$$
\text{model} = \text{NormalizingFlowModel}(\text{prior}, \text{flows})
$$

In [ ]:
# Define the prior distribution: 
# Logistic distribution via transformation of a Uniform
prior = TransformedDistribution(
    Uniform(torch.zeros(2, device=device), torch.ones(2, device=device)),
    SigmoidTransform().inv
)
# prior = MultivariateNormal(torch.zeros(2, device=device), torch.eye(2, device=device))

# Construct flows: Here we choose Neural Spline Flow Coupling layers (NSF_CL)
# You can experiment with different flows (MAF, IAF, RealNVP, etc.)
nfs_flow = NSF_CL  # alternatively, use NSF_AR for autoregressive variant
flows = [nfs_flow(dim=2, K=8, B=3, hidden_dim=16) for _ in range(3)]
# Insert invertible 1x1 convolutions and ActNorm layers before each flow
convs = [Invertible1x1Conv(dim=2) for _ in flows]
norms = [ActNorm(dim=2) for _ in flows]
# Interleave the layers
flows = list(itertools.chain(*zip(norms, convs, flows)))

# Construct the full normalizing flow model and move it to the device
model = NormalizingFlowModel(prior, flows).to(device)
print("Number of model parameters:", sum(p.numel() for p in model.parameters()))

## Optimizer and Training Setup

We use the Adam optimizer with a small weight decay to regularize the model. The loss function is the negative log-likelihood (NLL) computed from the flow:

$$
\text{loss} = -\sum \log p(z_K)
$$

where $z_K$ is the output of the final flow layer.

The training loop will sample a batch from the dataset, compute the transformed latent variables, and update the model parameters.

In [ ]:
# Set up optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

## Training the Normalizing Flow Model

We now train the model for a number of epochs. In each iteration:

1. Sample a batch of data.
2. Pass the data through the flow model to obtain:
   - Intermediate latent representations.
   - Log-probability of the prior.
   - Log-determinant of the Jacobians from the flow layers.
3. Compute the overall log-probability:

$$
\log p(x) = \log p(z_0) + \text{log\_det}
$$

4. Minimize the negative log-likelihood (NLL).

In [ ]:
n_epochs = 1000
batch_size = 128

model.train()
pbar = tqdm(range(n_epochs), desc='Training Loss')
for epoch in pbar:
    # Sample data and move to the selected device
    x_batch = d.sample(batch_size).to(device)
    
    # Forward pass: compute latent representations, prior log-prob, and log-determinant
    zs, prior_logprob, log_det = model(x_batch)
    logprob = prior_logprob + log_det  # overall log probability
    loss = -torch.sum(logprob)         # negative log-likelihood
    
    # Backward pass and optimization step
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    pbar.set_description(f"Loss: {loss.item():.2f}")
pbar.close()

## Evaluating and Visualizing the Model

After training, we evaluate the model in two ways:

1. **Data $\to$ Latent:** We pass data through the forward flow to see how the distribution transforms.
2. **Latent $\to$ Data:** We sample from the prior and use the inverse flow to generate data.

The following plots compare the transformed data with the prior distribution and the original data.

In [ ]:
# Set model to evaluation mode
model.eval()

# Sample a batch of data and move to device
x_eval = d.sample(128).to(device)
with torch.no_grad():
    zs, _, _ = model(x_eval)
    z_transformed = zs[-1]

# Move data back to CPU for plotting
x_np = x_eval.cpu().numpy()
z_np = z_transformed.cpu().numpy()
# Sample from the prior distribution (for visualization)
p_samples = model.prior.sample([128, 2]).squeeze().cpu().numpy()

plt.figure(figsize=(10, 5))
plt.subplot(121)
plt.scatter(p_samples[:, 0], p_samples[:, 1], c='g', s=5)
plt.scatter(z_np[:, 0], z_np[:, 1], c='r', s=5)
plt.legend(['Prior', 'Transformed (x → z)'])
plt.axis('equal')
plt.title("Data to Latent Space")

# Inverse transformation: generate samples from latent space to data space
zs_inverse = model.sample(128 * 8)
z_inv = zs_inverse[-1].detach().cpu().numpy()

plt.subplot(122)
plt.scatter(x_np[:, 0], x_np[:, 1], c='b', s=5, alpha=0.5)
plt.scatter(z_inv[:, 0], z_inv[:, 1], c='r', s=5, alpha=0.5)
plt.legend(['Real Data', 'Generated (z → x)'])
plt.axis('equal')
plt.title("Latent to Data Space")
plt.show()

## Visualizing Intermediate Flow Transformations

To gain insight into how the flow model gradually warps the space, we visualize the evolution of a grid through the inverse flow.

We start by constructing a grid of points in the 2D space and then pass these points backward through the flow layers.
At each step, we plot:

- The movement of points (using quiver plots).
- The warping of the grid lines.

This detailed visualization helps illustrate the contribution of each flow layer.

In [ ]:
from matplotlib import collections as mc

# Create a grid of points
ng = 20
xx, yy = np.linspace(-3, 3, ng), np.linspace(-3, 3, ng)
xv, yv = np.meshgrid(xx, yy)
xy_grid = np.stack([xv, yv], axis=-1)
in_circle = np.sqrt((xy_grid**2).sum(axis=2)) <= 3
xy_grid = xy_grid.reshape((ng * ng, 2))
xy_tensor = torch.from_numpy(xy_grid.astype(np.float32)).to(device)

# Pass the grid points backward through the flow
with torch.no_grad():
    zs_backward, _ = model.inverse(xy_tensor)

backward_flow_names = [type(f).__name__ for f in model.flow.flows[::-1]]
num_layers = len(zs_backward)
for i in range(num_layers - 1):
    z0 = zs_backward[i].cpu().numpy()
    z1 = zs_backward[i + 1].cpu().numpy()
    
    # Create subplots for point transitions and grid warp visualization
    fig, axs = plt.subplots(1, 2, figsize=(6, 3))
    
    # Plot the movement of points between consecutive layers
    axs[0].scatter(z0[:, 0], z0[:, 1], c='r', s=3, label='Before')
    axs[0].scatter(z1[:, 0], z1[:, 1], c='b', s=3, label='After')
    axs[0].quiver(z0[:, 0], z0[:, 1], z1[:, 0] - z0[:, 0], z1[:, 1] - z0[:, 1],
                  units='xy', scale=1, alpha=0.5)
    axs[0].axis([-3, 3, -3, 3])
    axs[0].set_title(f"Layer {i} → {i+1}\n({backward_flow_names[i]})")
    axs[0].legend()
    
    # Visualize grid warping at the current layer
    q = z1.reshape((ng, ng, 2))
    # Plot vertical grid lines
    p1 = np.reshape(q[1:, :, :], (ng**2 - ng, 2))
    p2 = np.reshape(q[:-1, :, :], (ng**2 - ng, 2))
    inc = np.reshape(in_circle[1:, :] | in_circle[:-1, :], (ng**2 - ng,))
    p1, p2 = p1[inc], p2[inc]
    line_collection_y = mc.LineCollection(zip(p1, p2), linewidths=1, alpha=0.5, color='k')
    # Plot horizontal grid lines
    p1 = np.reshape(q[:, 1:, :], (ng**2 - ng, 2))
    p2 = np.reshape(q[:, :-1, :], (ng**2 - ng, 2))
    inc = np.reshape(in_circle[:, 1:] | in_circle[:, :-1], (ng**2 - ng,))
    p1, p2 = p1[inc], p2[inc]
    line_collection_x = mc.LineCollection(zip(p1, p2), linewidths=1, alpha=0.5, color='k')
    
    axs[1].add_collection(line_collection_y)
    axs[1].add_collection(line_collection_x)
    axs[1].axis([-3, 3, -3, 3])
    axs[1].set_title(f"Grid Warp at End of Layer {i+1}")
    
    # Optionally, overlay the original data points
    axs[1].scatter(x_np[:, 0], x_np[:, 1], c='r', s=5, alpha=0.5)
    
    plt.tight_layout()
    plt.show()

## Additional Training and Dynamic Grid Visualization

In this section, we further train the model for a few iterations while dynamically visualizing the evolution of the grid warp. This is useful for understanding how the flow adjusts during training.

We use `matplotlib.gridspec` to organize multiple subplots that display:
- The point-wise transformation at different layers.
- The overall grid warping.

The loop runs for a limited number of steps and breaks after the first visualization update.

In [ ]:
ng = 20
xx, yy = np.linspace(-3, 3, ng), np.linspace(-3, 3, ng)
xv, yv = np.meshgrid(xx, yy)
xy_grid = np.stack([xv, yv], axis=-1)
in_circle = np.sqrt((xy_grid**2).sum(axis=2)) <= 3
xy_grid = xy_grid.reshape((ng * ng, 2))
xy_tensor = torch.from_numpy(xy_grid.astype(np.float32)).to(device)

# Validation data for overlaying on plots
x_val = d.sample(128 * 5).to(device)

model.train()
for k in range(500):
    # Sample a training batch
    x_batch = d.sample(128).to(device)
    
    # Forward pass
    zs, prior_logprob, log_det = model(x_batch)
    logprob = prior_logprob + log_det
    loss = -torch.sum(logprob)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if k % 10 == 0:
        with torch.no_grad():
            zs_backward, _ = model.inverse(xy_tensor)
        backward_flow_names = [type(f).__name__ for f in model.flow.flows[::-1]]
        num_layers = len(zs_backward)
        # Choose one layer index for visualization (e.g., second-to-last)
        layer_idx = num_layers - 2
        z0 = zs_backward[layer_idx].cpu().numpy()
        z1 = zs_backward[layer_idx + 1].cpu().numpy()
        
        ss = 0.1  # spacing for subplots
        fig = plt.figure(constrained_layout=True, figsize=(10, 5))
        outer = fig.add_gridspec(1, 2, wspace=ss, hspace=ss)
        inner1 = outer[0].subgridspec(3, 3, wspace=ss, hspace=ss)
        inner2 = outer[1].subgridspec(1, 1, wspace=ss, hspace=ss)
        
        # Plot intermediate transformations for up to 9 layers (or as many as available)
        for i in range(min(num_layers - 1, 9)):
            ax = plt.Subplot(fig, inner1[i])
            z_prev = zs_backward[i].cpu().numpy()
            z_next = zs_backward[i + 1].cpu().numpy()
            ax.scatter(z_prev[:, 0], z_prev[:, 1], c='r', s=1, alpha=0.5)
            ax.scatter(z_next[:, 0], z_next[:, 1], c='b', s=1, alpha=0.5)
            ax.quiver(z_prev[:, 0], z_prev[:, 1], z_next[:, 0] - z_prev[:, 0], z_next[:, 1] - z_prev[:, 1],
                      units='xy', scale=1, alpha=0.5)
            ax.axis([-3, 3, -3, 3])
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            fig.add_subplot(ax)
        
        # Plot grid warp for the chosen layer
        ax = plt.Subplot(fig, inner2[0])
        q = z1.reshape((ng, ng, 2))
        # Vertical grid lines
        p1 = np.reshape(q[1:, :, :], (ng**2 - ng, 2))
        p2 = np.reshape(q[:-1, :, :], (ng**2 - ng, 2))
        inc = np.reshape(in_circle[1:, :] | in_circle[:-1, :], (ng**2 - ng,))
        p1, p2 = p1[inc], p2[inc]
        lcy = mc.LineCollection(zip(p1, p2), linewidths=1, alpha=0.5, color='k')
        # Horizontal grid lines
        p1 = np.reshape(q[:, 1:, :], (ng**2 - ng, 2))
        p2 = np.reshape(q[:, :-1, :], (ng**2 - ng, 2))
        inc = np.reshape(in_circle[:, 1:] | in_circle[:, :-1], (ng**2 - ng,))
        p1, p2 = p1[inc], p2[inc]
        lcx = mc.LineCollection(zip(p1, p2), linewidths=1, alpha=0.5, color='k')
        ax.add_collection(lcy)
        ax.add_collection(lcx)
        ax.axis([-3, 3, -3, 3])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        fig.add_subplot(ax)
        
        # Overlay validation data on the grid warp plot
        plt.scatter(x_val[:, 0].cpu().numpy(), x_val[:, 1].cpu().numpy(), c='r', s=5, alpha=0.5)
        
        plt.suptitle(f"Dynamic Grid Warp Visualization")
        plt.show()
        break  # break after the first visualization update

## Visualizing the Final Density

In the final step, we evaluate the log-density over a fine grid and visualize the probability values.
This gives us an intuition about how the normalizing flow has shaped the density.

We compute:

$$
\hat{p}(x) = \exp\left(\log p(x)\right)
$$

and use a scatter plot where the color represents the density.

In [ ]:
ng = 100
xx, yy = np.linspace(-3, 3, ng), np.linspace(-3, 3, ng)
xv, yv = np.meshgrid(xx, yy)
xy_grid = np.stack([xv, yv], axis=-1)
xy_grid = xy_grid.reshape((ng * ng, 2))
xy_tensor = torch.from_numpy(xy_grid).float().to(device)

with torch.no_grad():
    _, prior_logprob, _ = model(xy_tensor)
density = prior_logprob.exp().cpu().numpy()

plt.figure(figsize=(6, 5))
plt.scatter(xy_grid[:, 0], xy_grid[:, 1], c=density, cmap='viridis', s=5)
plt.colorbar(label='Density')
plt.title("Final Estimated Density")
plt.axis('equal')
plt.show()